# Exercise 1: Maximising NumPy's Performance - Some Simple Tips

## Learning Objectives

This workbook will be focussed on getting the best performance out of NumPy, and some common performance pitfalls when mixing NumPy and pure Python.

Through the exercises, you will:
- Get familiar with the `%timeit` and `%%timeit` line/cell magics for timing in Jupyter Notebooks
- Learn how mixing NumPy functions with Python Lists can kill your performance
- Practice identifying and fixing these kinds of "mixing problems" in different scenarios

## Setup - Don't Forget To Run!

In [ ]:
import numpy as np
import statistics
import random

In [ ]:
PROBLEM_SIZE = 1_000_000

## Problem 1: Mixing NumPy Functions and Python Lists

In the lectures, we compared the performance of the following two functions which calculate the mean:

In [ ]:
def naive_mean(array):
  sum = 0
  for x in array:
    sum += x
  mean = sum / len(array)
  return mean
    
def np_mean(array):
  return np.mean(array)

We applied these functions to both Python Lists, and NumPy Arrays, to see how their performance would compare:

In [ ]:
x_list = list(range(PROBLEM_SIZE))
x_array = np.array(x_list)

# Time both functions with Python list
print("Naive Mean on Python List:")
%timeit naive_mean(x_list)
print("NumPy Mean on Python List:")
%timeit np_mean(x_list)

# Time both functions with NumPy Array
print("Naive Mean on NumPy Array:")
%timeit naive_mean(x_array)
print("NumPy Mean on NumPy Array:")
%timeit np_mean(x_array)

We noted a couple of things:
- When both operating on a list, the Python for loop beats the NumPy function
- When we let the NumPy mean operate on a NumPy array, it gets a **huge speedup** and finally beats the for loop

We discussed how this was due to the [fundamental differences between Python Lists and NumPy Arrays.](https://kiranjonathan-dev.github.io/ICSC26_Accelerating_Python_With_Numba_Lecture_Slides/47).

What we didn't have time to see, is that the Python for loop also slows down when operating on a NumPy Array instead of a Python List.

#### **Can you explain why?**

<details>
<summary>Reveal Explanation</summary>
As the array is stored as a simple C-style array (NumPy arrays are essentially pointers with some metadata), each element of the array has to be reconstructed into a Python float object as you loop through it. This adds a lot of overhead to the For Loop, which makes it slower!
</details>

### New Contenders - Python `sum()`

It wasn't an *entirely* fair comparison to show the NumPy mean against a raw Python For Loop, since there are many other ways to calculate the mean in Python, without using NumPy.

One way is using the inbuilt `sum()` method:

In [ ]:
def sum_mean(array):
    return sum(array) / len(array)

### Practical Exercise:

Implement similar timing tests with `%timeit` that compares each functions performance, using both a Python List and a NumPy Array as an input.

**Before you run your tests**, try to make a few predictions:
- Will `sum()` be faster on Python Lists or NumPy Arrays?
- Will `sum()` be faster than the For Loop was?
- Will `sum()` even be faster than NumPy mean?

**Make sure you use arrays of the same size if you're comparing with the For Loop/NumPy results. The arrays below already meet this, as long as you haven't changed the above**

In [ ]:
x_list = list(range(PROBLEM_SIZE))
x_array = np.array(x_list)

# Place your timings here, remember to run for both the list and the array!

### Exercise Solution:

In [ ]:
# SOLUTION - Click "..." to expand

x_list = list(range(PROBLEM_SIZE))
x_array = np.array(x_list)

# Time with Python list
print("sum() mean on Python List")
%timeit sum_mean(x_list)

# Time with NumPy Array
print("sum() mean on NumPy Array")
%timeit sum_mean(x_array)

#### **Explanation:**

<details>
<summary>Show Solution</summary>
Let's answer our original questions:
    
- Will `sum()` be faster on Python Lists or NumPy Arrays?
    - Python Lists
- Will `sum()` be faster than the For Loop was?
    - Yes!
- Will `sum()` even be faster than NumPy mean?
    - No (phew)

`sum()` is our new champion for operating on Python Lists, but it still can't be NumPy's time when operating on NumPy Arrays.

Python's in-built functions are highly optimised for Python Lists, and show once again **that we should generally avoid For Loops where other options are available**.

However, at the end of the day, **operations on Python Lists will never be able to beat NumPy operations on NumPy Arrays**, due to the limitations of Python Lists. As they are not contiguous in memory, Python's lists can't allow for performance gains like **SIMD (Single Instruction, Multiple Data) vectorisation**, or memory-friendly **cache prefetching!**  
</details>


## Problem 2: NumPy Array Initialisations

We found out in the lecture that these timings were slightly misleading, as they didn't include array creation.

Including the array creation, we get timings more like:

In [ ]:
%%timeit

x_list = list(range(PROBLEM_SIZE))
naive_mean(x_list)

In [ ]:
%%timeit

x_list = list(range(PROBLEM_SIZE))
x_array = np.array(x_list)
naive_mean(x_array)

We can see that we haven't actually gained any performance, we just shifted the slow part out of the timing zone!

From this we learn two important lessons:
- To make sure our timing is always comprehensive (you'd be surprised where the slow bits are!)
- To always prefer Numpy Array initialisation!

Here's an example where we swap our Python `range()` intialisation for a NumPy `np.arange()` intialisation that takes the same arguments/provides the same output:

In [ ]:
%%timeit

x_array = np.arange(PROBLEM_SIZE)
np_mean(x_array)

Now we finally get the speedup we were hoping for!

In this case, replacing `range()` with `np.arange()` makes sense, as they do the same thing. But what can we do with more complicated examples?

### Example Transformations From Pure Python to NumPy Arrays

Here are some examples of how we can transform common python list inialisations into NumPy array intialisations:

#### Pure Python - Range (i) Based Array

In [ ]:
%%timeit
x_list = []
for i in range(PROBLEM_SIZE):
    x_list.append(2*i + 5)

Or, equivalently (but slightly faster) with list comprehension:

In [ ]:
%%timeit
x_list = [2*i + 5 for i in range(PROBLEM_SIZE)]

#### NumPy - Range (i) Based Array

In [ ]:
%%timeit
x_array = np.arange(PROBLEM_SIZE) * 2 + 5

#### Pure Python - Uniform Array

In [ ]:
%%timeit
x_list = [10] * PROBLEM_SIZE

#### NumPy - Uniform Array

In [ ]:
%%timeit
x_array = np.ones(PROBLEM_SIZE) * 10

Or much more efficiently!

In [ ]:
%%timeit
x_array = np.full(PROBLEM_SIZE, 10)

#### Pure Python - Random Array

In [ ]:
%%timeit
x_list = []
for i in range(PROBLEM_SIZE):
    x_list.append(random.random())

#### NumPy - Random Array

In [ ]:
%%timeit
x_array = np.random.random(PROBLEM_SIZE)

Basically, NumPy has all kinds of ways to intialise arrays, and there's many different ways to combine them to get the same result!

Whilst we've been showing timings as an educational exercise, I wouldn't worry **too** much about your array initialisation, unless you've **already profiled and identified it as a hot spot!**

### Practical Exercises

Now that we've had a look at some of the ways to refactor list intialisations into NumPy Array intialisations, it's time for you to have a go!

For each of the exercises below, rewrite the array intialisations using only NumPy, **no Python For Loops!**

### Exercise A

In [ ]:
%%timeit
x_list = [np.sin(i**2)**2 for i in range(PROBLEM_SIZE)]

In [ ]:
%%timeit
# Your NumPy code here

### Solution A

In [ ]:
%%timeit

x_array = np.sin(np.arange(PROBLEM_SIZE)**2)**2

### Exercise B

In [ ]:
%%timeit
x_list = [np.cos(np.pi) for i in range(PROBLEM_SIZE)]

In [ ]:
%%timeit
# Your NumPy code here

### Solution B

In [ ]:
%%timeit

np.fill(np.cos(np.pi))

### Exercise C (Hard)

In [ ]:
%%timeit

x_list = []
for i in range(PROBLEM_SIZE):
    if i%2 == 0:
        x_list.append(i)
    else:
        x_list.append(0)

In [ ]:
%%timeit
# Your NumPy code here

### Solution C

In [ ]:
%%timeit

x_array = np.arange(PROBLEM_SIZE)
x_array = (x_array % 2) * x_array
print(x_array)

## Problem 3: General NumPy Refactors

For the final problem in this NumPy section, we will do some general exercises around identifying and fixing NumPy/pure Python mixing.

For each of the following code snippets, implement a new version of the function that makes use of NumPy functions and NumPy arrays, and time both to see what kind of performance gains you've achieved!

### Exercise A

In [ ]:
%%timeit
x_list = list(range(PROBLEM_SIZE))

max_x = max(x_list)
norm_to_max_list = [x / max_x for x in x_list]

In [ ]:
%%timeit
# Your NumPy code here

### Solution A

In [ ]:
%%timeit

x_array = np.arange(PROBLEM_SIZE)
norm_to_max_array = x_array / np.max(x_array)

### Exercise B

In [ ]:
%%timeit
x_list = list(range(PROBLEM_SIZE))

strange_sum = 0
for i in range(len(x_list)):
    if x_list[i] > 10:
        strange_sum += x_list[i]**2 + 2

In [ ]:
%%timeit
# Your NumPy code here

### Solution B

In [ ]:
%%timeit

x_array = np.arange(PROBLEM_SIZE)
strange_sum = np.sum(x_array[x_array > 10]**2+2)

### Exercise C

In [ ]:
%%timeit
x_list = [i/2 for i in range(PROBLEM_SIZE)]
y_list = [4] * PROBLEM_SIZE

dot_product = 0
for i in range(len(x_list)):
    dot_product += x_list[i] * y_list[i]

In [ ]:
%%timeit
# Your NumPy code here

### Solution C

In [ ]:
%%timeit
x_array = np.arange(PROBLEM_SIZE) / 2
y_array = np.full(PROBLEM_SIZE, 4)

dot_product = np.dot(x_array, y_array)

## Bonus Challenge (Very Difficult): Game of Life Rewrite! 
### (Do this at the end of the exercise class if you finish early)

#### Conway's Game of Life

**Conway's Game of Life** is a cellular automaton played on a 2D grid of cells.

Each cell is either **alive (1)** or **dead (0)**. The grid evolves in discrete time steps according to how many of a cell's neighbours are alive/dead.

The rules for each cell are as follows:
1. Any live cell with **fewer than 2** live neighbors dies (underpopulation).
2. Any live cell with **2 or 3** live neighbors survives.
3. Any live cell with **more than 3** live neighbors dies (overpopulation).
4. Any dead cell with **exactly 3** live neighbors becomes alive (reproduction).

Despite these simple rules, the system produces surprisingly complex patterns.

It's quite natural to write the game using a double for loop, as in the below code:

In [ ]:
def time_step(grid):
    rows = len(grid)
    cols = len(grid[0])

    new_grid = [[0]*cols for _ in range(rows)]

    for i in range(rows):
        for j in range(cols):
            neighbors = 0

            for di in [-1, 0, 1]:
                for dj in [-1, 0, 1]:
                    if di == 0 and dj == 0:
                        continue

                    ni = i + di
                    nj = j + dj

                    if 0 <= ni < rows and 0 <= nj < cols:
                        neighbors += grid[ni][nj]

            if grid[i][j] == 1 and neighbors in (2, 3):
                new_grid[i][j] = 1
            elif grid[i][j] == 0 and neighbors == 3:
                new_grid[i][j] = 1

    return new_grid

In [ ]:
%%timeit

# Problem size
rows = 100
cols = 100
num_steps = 100

# Intialise array
grid_list = []
for i in range(rows):
    row_list = []
    
    for j in range(cols):
        row_list.append(round(random.random()))

    grid_list.append(row_list)

# Perform calculations
for i in range(num_steps):
    grid_list = time_step(grid_list)

Can you rewrite both the array intialisation and the time step function using only NumPy?

Here's a hint, this example may violate the rule that values depend on neighbouring values, but because of the periodic boundaries we can still use NumPy! You might just need the function `np.roll()`, which you can read about [here](https://www.geeksforgeeks.org/python/numpy-roll-python/).

In [ ]:
# Your rewrite goes here


#### Solution

In [ ]:
# Solution

def time_step_np(grid):
    neighbors = (
        np.roll(grid, 1, 0) + np.roll(grid, -1, 0) +
        np.roll(grid, 1, 1) + np.roll(grid, -1, 1) +
        np.roll(np.roll(grid, 1, 0), 1, 1) +
        np.roll(np.roll(grid, 1, 0), -1, 1) +
        np.roll(np.roll(grid, -1, 0), 1, 1) +
        np.roll(np.roll(grid, -1, 0), -1, 1)
    )

    return ((neighbors == 3) | ((grid == 1) & (neighbors == 2))).astype(int)

# Problem size
rows = 5
cols = 5
num_steps = 100

# Initialise array
grid = np.random.randint(0, 2, (rows, cols))

# Perform calculations
for i in range(num_steps):
    grid = time_step_np(grid)